In [2]:
!nvidia-smi

Wed Jun 17 23:38:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import os, random, warnings, json
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix, roc_curve
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

# ─── Reproductibilité ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ─── Configuration ────────────────────────────────────────────────────────────
CFG = {
    # Dataset
    "csv_path": "asthma_disease_data.csv",  # adapter si nécessaire
    # MDP
    "n_actions": 5,        # Paliers GINA 1–5
    "gamma_discount": 0.95,
    # Récompense
    "alpha": 0.55,         # poids contrôle
    "beta":  0.35,         # poids sécurité
    "gamma_med": 0.10,     # poids médication
    "exacer_penalty": -6.0,
    "control_bonus": +2.5,
    # DQN
    "buffer_size": 30_000,
    "batch_size": 64,
    "lr": 1e-3,
    "target_update": 100,
    "n_episodes": 600,
    "max_steps": 40,
    "eps_start": 1.0,
    "eps_end": 0.01,
    "eps_decay": 0.994,
    "hidden": [256, 128, 64],
    # Digital Twin
    "twin_horizon": 30,     # jours simulés par épisode
    "twin_noise": 0.05,     # bruit stochastique du twin
}

# ─── Features du dataset ─────────────────────────────────────────────────────
FEATURE_COLS = [
    "Age", "Gender", "Ethnicity", "EducationLevel", "BMI", "Smoking",
    "PhysicalActivity", "DietQuality", "SleepQuality",
    "PollutionExposure", "PollenExposure", "DustExposure",
    "PetAllergy", "FamilyHistoryAsthma", "HistoryOfAllergies",
    "Eczema", "HayFever", "GastroesophagealReflux",
    "LungFunctionFEV1", "LungFunctionFVC",
    "Wheezing", "ShortnessOfBreath", "ChestTightness",
    "Coughing", "NighttimeSymptoms", "ExerciseInduced",
]

# Actions → effet sur les features modifiables (simulation)
GINA_ACTIONS = {
    0: {"name": "Palier 1 – Formotérol/Budésonide à la demande", "fev1_delta": +0.05,  "symptom_relief": 0.30},
    1: {"name": "Palier 2 – CSI faible dose + SABA",            "fev1_delta": +0.10,  "symptom_relief": 0.50},
    2: {"name": "Palier 3 – CSI faible dose + LABA",            "fev1_delta": +0.15,  "symptom_relief": 0.65},
    3: {"name": "Palier 4 – CSI forte dose + LABA",             "fev1_delta": +0.18,  "symptom_relief": 0.75},
    4: {"name": "Palier 5 – Biothérapie / Corticoïdes oraux",   "fev1_delta": +0.20,  "symptom_relief": 0.85},
}


# ══════════════════════════════════════════════════════════════════════════════
# 1. CHARGEMENT & EDA
# ══════════════════════════════════════════════════════════════════════════════

def load_and_eda(csv_path: str) -> pd.DataFrame:
    """Charge le CSV, affiche les statistiques clés, retourne le DataFrame."""
    df = pd.read_csv(r"/content/asthma_disease_data.csv")
    print("=" * 65)
    print(" DATASET : asthma_disease_data.csv")
    print("=" * 65)
    print(f"  Lignes : {df.shape[0]:,}  |  Colonnes : {df.shape[1]}")
    print(f"  Valeurs manquantes : {df.isnull().sum().sum()}")
    print(f"  Asthmatiques (Diagnosis=1) : {df['Diagnosis'].sum()} "
          f"({df['Diagnosis'].mean()*100:.1f}%)")
    print(f"  Non-asthmatiques           : {(df['Diagnosis']==0).sum()} "
          f"({(df['Diagnosis']==0).mean()*100:.1f}%)")
    print("\n  Top features corrélées au diagnostic :")
    corr = df[FEATURE_COLS + ["Diagnosis"]].corr()["Diagnosis"] \
             .drop("Diagnosis").abs().sort_values(ascending=False)
    for feat, val in corr.head(8).items():
        print(f"    {feat:<30}  {val:.4f}")
    print("=" * 65)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 2. MODÈLE DE RISQUE D'ASTHME
# ══════════════════════════════════════════════════════════════════════════════

class AsthmaRiskModel:
    """
    Modèle de classification binaire (asthme / pas d'asthme).
    RandomForest + SMOTE pour le déséquilibre de classes.
    Fournit :
      - predict_proba(X) → probabilité de risque ∈ [0,1]
      - score() → AUC, rapport de classification
    """

    def __init__(self):
        self.model = RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_leaf=4,
            class_weight="balanced", random_state=SEED, n_jobs=-1
        )
        self.scaler = MinMaxScaler()
        self.feature_importances_: Optional[pd.Series] = None
        self.auc_: float = 0.0
        self._fitted = False

    def fit(self, df: pd.DataFrame) -> Dict:
        """Entraîne le modèle sur le dataset complet (avec SMOTE)."""
        X = df[FEATURE_COLS].values
        y = df["Diagnosis"].values

        # SMOTE — sur-échantillonnage de la classe minoritaire
        X_res, y_res = SMOTE(random_state=SEED, k_neighbors=5).fit_resample(X, y)
        X_scaled = self.scaler.fit_transform(X_res)

        X_tr, X_te, y_tr, y_te = train_test_split(
            X_scaled, y_res, test_size=0.2, stratify=y_res, random_state=SEED
        )
        self.model.fit(X_tr, y_tr)

        y_pred  = self.model.predict(X_te)
        y_proba = self.model.predict_proba(X_te)[:, 1]
        self.auc_ = roc_auc_score(y_te, y_proba)

        self.feature_importances_ = pd.Series(
            self.model.feature_importances_, index=FEATURE_COLS
        ).sort_values(ascending=False)

        self._fitted = True
        print(f"\n[RISK MODEL] RandomForest entraîné")
        print(f"  AUC ROC   : {self.auc_:.4f}")
        print(f"  Accuracy  : {(y_pred == y_te).mean():.4f}")
        print(f"  Top 5 features : {list(self.feature_importances_.head(5).index)}")

        metrics = {"auc": self.auc_,
                   "report": classification_report(y_te, y_pred, output_dict=True),
                   "cm": confusion_matrix(y_te, y_pred),
                   "fpr": roc_curve(y_te, y_proba)[0],
                   "tpr": roc_curve(y_te, y_proba)[1]}
        return metrics

    def predict_risk(self, x: np.ndarray) -> float:
        """Retourne la probabilité de risque d'asthme pour un état patient."""
        assert self._fitted, "Modèle non entraîné"
        x_scaled = self.scaler.transform(x.reshape(1, -1))
        return float(self.model.predict_proba(x_scaled)[0, 1])


# ══════════════════════════════════════════════════════════════════════════════
# 3. DIGITAL TWIN DU PATIENT
# ══════════════════════════════════════════════════════════════════════════════

class PatientDigitalTwin:
    """
    Jumeau numérique (Digital Twin) d'un patient asthmatique.

    Principe :
      - Initialisé avec le profil réel d'un patient du dataset
      - Simule l'évolution de ses paramètres physiologiques sur un horizon T
        en réponse aux actions thérapeutiques
      - Le modèle de risque RF évalue à chaque pas le risque d'exacerbation
      - La dynamique est partiellement stochastique pour refléter
        la variabilité inter-journalière réelle

    Indices des features clés dans FEATURE_COLS :
      FEV1  : 18   FVC   : 19
      Wheez : 20   SoB   : 21   Tight : 22
      Cough : 23   Night : 24   Exind : 25
      Smok  : 5    PhysAct: 6
    """

    IDX = {feat: i for i, feat in enumerate(FEATURE_COLS)}

    def __init__(self, patient_row: np.ndarray, risk_model: AsthmaRiskModel,
                 noise: float = 0.05):
        self.base_state   = patient_row.copy().astype(np.float32)
        self.state        = patient_row.copy().astype(np.float32)
        self.risk_model   = risk_model
        self.noise        = noise
        self.gina_step    = 1         # palier thérapeutique actuel
        self.t            = 0
        self.exacerbations = 0

    def reset(self) -> np.ndarray:
        self.state = self.base_state.copy()
        # Légère variation aléatoire des conditions initiales
        rng = np.random
        self.state[self.IDX["PollutionExposure"]] = np.clip(
            self.base_state[self.IDX["PollutionExposure"]] + rng.normal(0, 0.5), 0, 10
        )
        self.state[self.IDX["PollenExposure"]] = np.clip(
            self.base_state[self.IDX["PollenExposure"]] + rng.normal(0, 0.5), 0, 10
        )
        self.gina_step = 1
        self.t = 0
        self.exacerbations = 0
        return self._get_obs()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, dict]:
        """
        Applique l'action (palier GINA) et simule un pas de temps (1 jour).
        Retourne : (next_obs, reward, done, info)
        """
        action_info = GINA_ACTIONS[action]
        self.gina_step = action + 1
        noise = np.random.normal(0, self.noise, len(FEATURE_COLS))

        # ── Dynamique physiologique ───────────────────────────────────────
        # FEV1 s'améliore avec le traitement
        fev1_idx = self.IDX["LungFunctionFEV1"]
        self.state[fev1_idx] = np.clip(
            self.state[fev1_idx] + action_info["fev1_delta"] + noise[fev1_idx],
            0.8, 4.5
        )

        # Symptômes s'améliorent selon le taux de soulagement du palier
        sr = action_info["symptom_relief"]
        for sym in ["Wheezing", "ShortnessOfBreath", "ChestTightness",
                    "Coughing", "NighttimeSymptoms", "ExerciseInduced"]:
            idx = self.IDX[sym]
            # Tendance vers 0 (pas de symptôme) modulée par relief + bruit
            self.state[idx] = np.clip(
                self.state[idx] * (1 - sr * 0.15) + noise[idx] * 0.1, 0, 1
            )

        # Variables environnementales : évolution aléatoire saisonnière
        for env_f in ["PollutionExposure", "PollenExposure", "DustExposure"]:
            idx = self.IDX[env_f]
            self.state[idx] = np.clip(self.state[idx] + np.random.normal(0, 0.3), 0, 10)

        self.t += 1

        # ── Score de contrôle simulé (proxy ACT, ∈ [0,25]) ───────────────
        fev1_norm = self.state[fev1_idx] / 4.0         # normalisé [0,1]
        symptom_burden = np.mean([
            self.state[self.IDX[s]]
            for s in ["Wheezing","ShortnessOfBreath","ChestTightness","Coughing"]
        ])
        act_proxy = 25 * (0.6 * fev1_norm + 0.4 * (1 - symptom_burden))
        act_proxy = float(np.clip(act_proxy + np.random.normal(0, 1.5), 0, 25))

        # ── Probabilité de risque via le Digital Twin (modèle RF) ─────────
        risk_score = self.risk_model.predict_risk(self.state)

        # ── Exacerbation simulée ──────────────────────────────────────────
        exacer_prob = risk_score * (1 - sr * 0.5)
        exacerbation = int(np.random.random() < exacer_prob * 0.3)
        if exacerbation:
            self.exacerbations += 1
            self.state[fev1_idx] = max(0.8, self.state[fev1_idx] - 0.3)

        # ── Récompense multi-objectif ─────────────────────────────────────
        r_control  = act_proxy / 25.0 + (CFG["control_bonus"] if act_proxy >= 20 else 0)
        r_safety   = CFG["exacer_penalty"] * exacerbation
        r_med      = -0.5 * (self.gina_step / 5.0)
        reward = (CFG["alpha"]     * r_control
                  + CFG["beta"]    * r_safety
                  + CFG["gamma_med"] * r_med)

        # ── Contrainte de sécurité : step-down brutal interdit ────────────
        if (self.gina_step - action) > 1:
            reward -= 1.0

        done = (self.t >= CFG["twin_horizon"])
        info = {
            "act_proxy": act_proxy,
            "risk_score": risk_score,
            "exacerbation": exacerbation,
            "gina_step": self.gina_step,
            "fev1": self.state[fev1_idx],
        }
        return self._get_obs(), reward, done, info

    def _get_obs(self) -> np.ndarray:
        """Vecteur d'observation normalisé (même échelle que le scaler RF)."""
        return self.risk_model.scaler.transform(
            self.state.reshape(1, -1)
        ).flatten().astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# 4. PRIORITIZED REPLAY BUFFER
# ══════════════════════════════════════════════════════════════════════════════

class PrioritizedReplayBuffer:
    """Prioritized Experience Replay (Schaul et al., 2016)."""

    def __init__(self, capacity: int, alpha: float = 0.6):
        self.capacity  = capacity
        self.alpha     = alpha
        self.buffer    = []
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.pos       = 0

    def push(self, s, a, r, s_, done):
        max_p = self.priorities.max() if self.buffer else 1.0
        item  = (torch.FloatTensor(s), torch.LongTensor([a]),
                 torch.FloatTensor([r]), torch.FloatTensor(s_),
                 torch.FloatTensor([float(done)]))
        if len(self.buffer) < self.capacity:
            self.buffer.append(item)
        else:
            self.buffer[self.pos] = item
        self.priorities[self.pos] = max_p
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int, beta: float = 0.4):
        n = len(self.buffer)
        probs = self.priorities[:n] ** self.alpha
        probs /= probs.sum()
        idx = np.random.choice(n, batch_size, p=probs, replace=False)
        weights = (n * probs[idx]) ** (-beta)
        weights /= weights.max()
        batch = [self.buffer[i] for i in idx]
        s  = torch.stack([b[0] for b in batch])
        a  = torch.stack([b[1] for b in batch]).squeeze(1)
        r  = torch.stack([b[2] for b in batch]).squeeze(1)
        s_ = torch.stack([b[3] for b in batch])
        d  = torch.stack([b[4] for b in batch]).squeeze(1)
        return (s, a, r, s_, d), idx, torch.FloatTensor(weights)

    def update_priorities(self, indices, td_errors):
        for i, e in zip(indices, td_errors):
            self.priorities[i] = abs(e) + 1e-6

    def __len__(self): return len(self.buffer)


# ══════════════════════════════════════════════════════════════════════════════
# 5. RÉSEAU Q + AGENT DOUBLE DQN
# ══════════════════════════════════════════════════════════════════════════════

class QNetwork(nn.Module):
    def __init__(self, state_dim: int, n_actions: int):
        super().__init__()
        h = CFG["hidden"]
        layers = []
        in_d = state_dim
        for hh in h:
            layers += [nn.Linear(in_d, hh), nn.BatchNorm1d(hh), nn.ReLU(), nn.Dropout(0.1)]
            in_d = hh
        layers.append(nn.Linear(in_d, n_actions))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


class DoubleDQNAgent:
    """
    Double DQN avec Prioritized Experience Replay.
    Séparation sélection (réseau online) / évaluation (réseau target)
    → réduit le biais de surestimation des Q-valeurs.
    """

    def __init__(self, state_dim: int, n_actions: int):
        self.n_actions = n_actions
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.q_on  = QNetwork(state_dim, n_actions).to(self.device)
        self.q_tgt = QNetwork(state_dim, n_actions).to(self.device)
        self.q_tgt.load_state_dict(self.q_on.state_dict()); self.q_tgt.eval()
        self.opt   = optim.Adam(self.q_on.parameters(), lr=CFG["lr"])
        self.buf   = PrioritizedReplayBuffer(CFG["buffer_size"])
        self.eps   = CFG["eps_start"]
        self.steps = 0
        self.losses: List[float] = []
        n_p = sum(p.numel() for p in self.q_on.parameters())
        print(f"\n[AGENT] Double DQN | state_dim={state_dim} | device={self.device} | params={n_p:,}")

    def act(self, state: np.ndarray, train: bool = True) -> int:
        if train and random.random() < self.eps:
            return random.randrange(self.n_actions)
        t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        self.q_on.eval()
        with torch.no_grad(): q = self.q_on(t).squeeze(0).cpu().numpy()
        self.q_on.train()
        return int(np.argmax(q))

    def store(self, s, a, r, s_, done): self.buf.push(s, a, r, s_, done)

    def learn(self) -> Optional[float]:
        if len(self.buf) < CFG["batch_size"]: return None
        beta = min(1.0, 0.4 + self.steps * 6e-6)
        (s, a, r, s_, d), idx, w = self.buf.sample(CFG["batch_size"], beta)
        s, a, r, s_, d, w = (t.to(self.device) for t in (s, a, r, s_, d, w))

        with torch.no_grad():
            best_a = self.q_on(s_).argmax(1)
            q_next = self.q_tgt(s_).gather(1, best_a.unsqueeze(1)).squeeze(1)
            target = r + CFG["gamma_discount"] * q_next * (1 - d)

        q_pred = self.q_on(s).gather(1, a.unsqueeze(1)).squeeze(1)
        td_err = (target - q_pred).detach().cpu().numpy()
        self.buf.update_priorities(idx, td_err)

        loss = (w * F.smooth_l1_loss(q_pred, target, reduction="none")).mean()
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q_on.parameters(), 1.0)
        self.opt.step()
        self.steps += 1
        if self.steps % CFG["target_update"] == 0:
            self.q_tgt.load_state_dict(self.q_on.state_dict())
        self.eps = max(CFG["eps_end"], self.eps * CFG["eps_decay"])
        lv = loss.item(); self.losses.append(lv); return lv

    def save(self, path):
        torch.save({"q_on": self.q_on.state_dict(), "eps": self.eps,
                    "steps": self.steps}, path)
        print(f"[AGENT] Sauvegardé → {path}")


# ══════════════════════════════════════════════════════════════════════════════
# 6. BASELINE GINA
# ══════════════════════════════════════════════════════════════════════════════

class GINABaseline:
    """Politique GINA 2023 : step-up si ACT < 19, step-down si contrôle prolongé."""

    def __init__(self): self.ctrl_streak = 0; self.step = 1

    def reset(self): self.ctrl_streak = 0; self.step = 1

    def act(self, obs: np.ndarray, act_proxy: float) -> int:
        if act_proxy >= 20:
            self.ctrl_streak += 1
            if self.ctrl_streak >= 3:
                self.step = max(1, self.step - 1)
        else:
            self.ctrl_streak = 0
            if act_proxy < 15:
                self.step = min(5, self.step + 1)
        return self.step - 1


# ══════════════════════════════════════════════════════════════════════════════
# 7. ENTRAÎNEMENT
# ══════════════════════════════════════════════════════════════════════════════

def train(agent: DoubleDQNAgent, df: pd.DataFrame,
          risk_model: AsthmaRiskModel, n_episodes: int = None):
    if n_episodes is None: n_episodes = CFG["n_episodes"]

    # On entraîne sur les patients asthmatiques uniquement
    asthma_patients = df[df["Diagnosis"] == 1][FEATURE_COLS].values
    if len(asthma_patients) == 0:
        asthma_patients = df[FEATURE_COLS].values

    hist = {"ep_reward": [], "ep_control": [], "ep_exacer": [], "eps": [], "loss": []}
    print(f"\n[TRAIN] {n_episodes} épisodes | {len(asthma_patients)} patients asthmatiques")

    for ep in range(n_episodes):
        row = asthma_patients[random.randint(0, len(asthma_patients)-1)]
        twin = PatientDigitalTwin(row, risk_model, noise=CFG["twin_noise"])
        obs  = twin.reset()

        ep_r, ep_ctrl, ep_exacer, ep_losses = 0., [], [], []

        for _ in range(CFG["max_steps"]):
            a = agent.act(obs, train=True)
            next_obs, rew, done, info = twin.step(a)
            agent.store(obs, a, rew, next_obs, done)
            loss = agent.learn()
            if loss: ep_losses.append(loss)

            ep_r += rew
            ep_ctrl.append(info["act_proxy"])
            ep_exacer.append(info["exacerbation"])
            obs = next_obs
            if done: break

        hist["ep_reward"].append(ep_r)
        hist["ep_control"].append(np.mean(ep_ctrl))
        hist["ep_exacer"].append(np.mean(ep_exacer))
        hist["eps"].append(agent.eps)
        hist["loss"].append(np.mean(ep_losses) if ep_losses else 0.)

        if (ep + 1) % 100 == 0:
            w = 100
            print(f"  Ép {ep+1:4d}/{n_episodes} | "
                  f"R_moy={np.mean(hist['ep_reward'][-w:]):+.2f} | "
                  f"ACT={np.mean(hist['ep_control'][-w:]):.1f} | "
                  f"Exacer={np.mean(hist['ep_exacer'][-w:]):.3f} | "
                  f"ε={agent.eps:.3f}")

    print("[TRAIN] Terminé.")
    return hist


# ══════════════════════════════════════════════════════════════════════════════
# 8. ÉVALUATION COMPARATIVE
# ══════════════════════════════════════════════════════════════════════════════

def evaluate(agent: DoubleDQNAgent, df: pd.DataFrame,
             risk_model: AsthmaRiskModel, n_patients: int = 50) -> dict:
    """Compare l'agent RL vs la baseline GINA sur n_patients de test."""
    device = next(agent.q_on.parameters()).device
    test_pts = df[FEATURE_COLS].values
    sample_idx = np.random.choice(len(test_pts), min(n_patients, len(test_pts)), replace=False)

    rl  = {"rewards": [], "act": [], "exacer": [], "steps": []}
    gna = {"rewards": [], "act": [], "exacer": [], "steps": []}
    baseline = GINABaseline()

    agent.q_on.eval()
    for idx in sample_idx:
        row = test_pts[idx]

        # ── RL agent ────────────────────────────────────────────────────
        twin = PatientDigitalTwin(row, risk_model); obs = twin.reset()
        ep_r, ep_act, ep_ex, ep_stp = [], [], [], []
        for _ in range(CFG["twin_horizon"]):
            with torch.no_grad():
                q = agent.q_on(torch.FloatTensor(obs).unsqueeze(0).to(device))
            a = q.argmax(1).item()
            obs, r, done, info = twin.step(a)
            ep_r.append(r); ep_act.append(info["act_proxy"])
            ep_ex.append(info["exacerbation"]); ep_stp.append(info["gina_step"])
            if done: break
        rl["rewards"].append(np.sum(ep_r)); rl["act"].append(np.mean(ep_act))
        rl["exacer"].append(np.mean(ep_ex)); rl["steps"].append(np.mean(ep_stp))

        # ── GINA baseline ───────────────────────────────────────────────
        twin2 = PatientDigitalTwin(row, risk_model); obs2 = twin2.reset()
        baseline.reset(); ep_r2, ep_act2, ep_ex2, ep_stp2 = [], [], [], []
        for _ in range(CFG["twin_horizon"]):
            act_proxy_tmp = (np.mean([obs2[i] for i in [20,21,22,23]]) * -25 + 25)
            a2 = baseline.act(obs2, act_proxy_tmp)
            obs2, r2, done2, info2 = twin2.step(a2)
            ep_r2.append(r2); ep_act2.append(info2["act_proxy"])
            ep_ex2.append(info2["exacerbation"]); ep_stp2.append(info2["gina_step"])
            if done2: break
        gna["rewards"].append(np.sum(ep_r2)); gna["act"].append(np.mean(ep_act2))
        gna["exacer"].append(np.mean(ep_ex2)); gna["steps"].append(np.mean(ep_stp2))

    agent.q_on.train()

    results = {}
    for k in ["rewards", "act", "exacer", "steps"]:
        results[f"rl_{k}"]  = float(np.mean(rl[k]))
        results[f"gna_{k}"] = float(np.mean(gna[k]))

    labels = {"rewards": "Récompense cumulée",
              "act":     "Score ACT proxy (0–25)",
              "exacer":  "Taux exacerbations",
              "steps":   "Palier GINA moyen"}
    print("\n" + "="*62)
    print("  ÉVALUATION OFFLINE — Agent RL vs Baseline GINA")
    print("="*62)
    print(f"  {'Métrique':<28} {'RL Agent':>12} {'GINA':>12}  {'Δ':>8}")
    print("-"*62)
    for k in ["rewards", "act", "exacer", "steps"]:
        rv = results[f"rl_{k}"]; gv = results[f"gna_{k}"]
        delta = rv - gv
        sign  = "▲" if delta > 0 and k != "exacer" else ("▼" if delta < 0 and k == "exacer" else "")
        print(f"  {labels[k]:<28} {rv:>12.3f} {gv:>12.3f} {delta:>+8.3f} {sign}")
    print("="*62)
    return results


# ══════════════════════════════════════════════════════════════════════════════
# 9. VISUALISATIONS
# ══════════════════════════════════════════════════════════════════════════════

def plot_eda(df: pd.DataFrame, out: str):
    """EDA : distribution, corrélations, profil asthmatiques."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Exploratory Data Analysis — Asthma Disease Dataset",
                 fontsize=14, fontweight="bold")

    # 1. Distribution Diagnosis
    ax = axes[0, 0]
    counts = df["Diagnosis"].value_counts()
    ax.bar(["Non-asthmatique\n(0)", "Asthmatique\n(1)"],
           [counts[0], counts.get(1, 0)], color=["#64B5F6", "#EF5350"])
    ax.set_title("Distribution du diagnostic"); ax.set_ylabel("Effectif")
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f"{int(bar.get_height())}", ha="center", fontsize=10)

    # 2. Age distribution par Diagnosis
    ax = axes[0, 1]
    for label, color, name in [(0, "#64B5F6", "Non-asthmatique"), (1, "#EF5350", "Asthmatique")]:
        ax.hist(df[df["Diagnosis"]==label]["Age"], bins=20, alpha=0.6,
                color=color, label=name, edgecolor="white")
    ax.set_title("Distribution de l'âge par diagnostic"); ax.set_xlabel("Âge")
    ax.legend(); ax.set_ylabel("Effectif")

    # 3. FEV1 vs FVC coloré par Diagnosis
    ax = axes[0, 2]
    for label, color, name in [(0, "#64B5F6", "Non-asthmatique"), (1, "#EF5350", "Asthmatique")]:
        sub = df[df["Diagnosis"]==label]
        ax.scatter(sub["LungFunctionFEV1"], sub["LungFunctionFVC"],
                   alpha=0.3, s=10, color=color, label=name)
    ax.set_xlabel("FEV1 (L)"); ax.set_ylabel("FVC (L)")
    ax.set_title("FEV1 vs FVC par diagnostic"); ax.legend()

    # 4. Top 10 feature importances
    ax = axes[1, 0]
    risk_tmp = RandomForestClassifier(n_estimators=100, random_state=SEED)
    X = df[FEATURE_COLS].values; y = df["Diagnosis"].values
    X_r, y_r = SMOTE(random_state=SEED).fit_resample(X, y)
    risk_tmp.fit(X_r, y_r)
    fi = pd.Series(risk_tmp.feature_importances_, index=FEATURE_COLS).sort_values()[-10:]
    fi.plot(kind="barh", ax=ax, color="#42A5F5")
    ax.set_title("Top 10 features (importance RF)"); ax.set_xlabel("Importance")

    # 5. Heatmap corrélations
    ax = axes[1, 1]
    sym_cols = ["LungFunctionFEV1", "LungFunctionFVC", "Wheezing",
                "ShortnessOfBreath", "ChestTightness", "Coughing",
                "NighttimeSymptoms", "ExerciseInduced", "Diagnosis"]
    corr_mat = df[sym_cols].corr()
    sns.heatmap(corr_mat, ax=ax, cmap="coolwarm", center=0,
                annot=True, fmt=".2f", linewidths=0.5, annot_kws={"size": 7})
    ax.set_title("Corrélations features clés")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

    # 6. Taux symptômes asthmatiques vs non
    ax = axes[1, 2]
    sym_feats = ["Wheezing", "ShortnessOfBreath", "ChestTightness",
                 "Coughing", "NighttimeSymptoms", "ExerciseInduced"]
    rates_ast  = df[df["Diagnosis"]==1][sym_feats].mean()
    rates_non  = df[df["Diagnosis"]==0][sym_feats].mean()
    x = np.arange(len(sym_feats)); w = 0.35
    ax.bar(x - w/2, rates_non, w, label="Non-asthmatique", color="#64B5F6")
    ax.bar(x + w/2, rates_ast,  w, label="Asthmatique",     color="#EF5350")
    ax.set_xticks(x); ax.set_xticklabels([s[:5] for s in sym_feats], rotation=45, fontsize=8)
    ax.set_title("Taux de symptômes par groupe"); ax.set_ylabel("Proportion")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[PLOT] EDA → {out}")


def plot_risk_model(metrics: dict, out: str):
    """Courbe ROC + matrice de confusion."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Modèle de Risque d'Asthme — RandomForest + SMOTE",
                 fontsize=13, fontweight="bold")

    # ROC
    ax1.plot(metrics["fpr"], metrics["tpr"], color="#EF5350", lw=2,
             label=f"ROC (AUC = {metrics['auc']:.4f})")
    ax1.plot([0,1],[0,1],"--", color="#AAAAAA", label="Aléatoire")
    ax1.set_xlabel("Taux faux positifs"); ax1.set_ylabel("Taux vrais positifs")
    ax1.set_title("Courbe ROC"); ax1.legend(); ax1.grid(alpha=0.3)

    # Confusion matrix
    cm = metrics["cm"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax2,
                xticklabels=["Prédit 0","Prédit 1"],
                yticklabels=["Réel 0","Réel 1"])
    ax2.set_title("Matrice de confusion (test set)")

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[PLOT] Risk model → {out}")


def plot_training(hist: dict, out: str):
    """Courbes d'apprentissage de l'agent RL."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("Apprentissage de l'Agent Double DQN — Digital Twin Asthme",
                 fontsize=13, fontweight="bold")

    def smooth(x, w=30): return pd.Series(x).rolling(w, min_periods=1).mean()

    configs = [
        (axes[0,0], hist["ep_reward"], "Récompense cumulée", "#2196F3"),
        (axes[0,1], hist["ep_control"], "Score ACT proxy moyen", "#4CAF50"),
        (axes[1,0], hist["ep_exacer"], "Taux d'exacerbations", "#F44336"),
        (axes[1,1], hist["loss"], "Loss (Huber)", "#9C27B0"),
    ]
    for ax, data, title, color in configs:
        ax.plot(data, alpha=0.2, color=color)
        ax.plot(smooth(data), color=color, lw=2)
        ax.set_title(title); ax.set_xlabel("Épisode"); ax.grid(alpha=0.3)
        ax.spines[["top","right"]].set_visible(False)

    # Epsilon overlay sur reward
    ax2 = axes[0,0].twinx()
    ax2.plot(hist["eps"], color="orange", lw=1, alpha=0.5, linestyle="--")
    ax2.set_ylabel("ε", color="orange"); ax2.tick_params(axis="y", labelcolor="orange")

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[PLOT] Training curves → {out}")


def plot_comparison(results: dict, out: str):
    """Graphique de comparaison RL vs GINA."""
    metrics = {
        "Récompense\ncumulée":        ("rewards", True),
        "Score ACT\nproxy (0–25)":    ("act",     True),
        "Taux d'\nexacerbations":     ("exacer",  False),
        "Palier GINA\nmoyen":         ("steps",   False),
    }
    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    fig.suptitle("Comparaison Agent RL vs Baseline GINA — Digital Twin",
                 fontsize=13, fontweight="bold")

    for ax, (label, (key, higher_better)) in zip(axes, metrics.items()):
        rl_v  = results[f"rl_{key}"]
        gna_v = results[f"gna_{key}"]
        better = (rl_v > gna_v) if higher_better else (rl_v < gna_v)
        colors = ["#4CAF50" if better else "#2196F3", "#BDBDBD"]
        bars = ax.bar(["RL Agent", "GINA"], [rl_v, gna_v],
                      color=colors, edgecolor="white", width=0.5)
        for bar, v in zip(bars, [rl_v, gna_v]):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height() + 0.005*max(abs(rl_v), abs(gna_v)),
                    f"{v:.3f}", ha="center", fontsize=10)
        ax.set_title(label, fontsize=11)
        ymax = max(abs(rl_v), abs(gna_v)) * 1.3
        ax.set_ylim(min(0, min(rl_v, gna_v)*1.3), ymax)
        ax.spines[["top","right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[PLOT] Comparison → {out}")


def plot_digital_twin_demo(df: pd.DataFrame, agent: DoubleDQNAgent,
                           risk_model: AsthmaRiskModel, out: str):
    """Simulation d'un patient sur 30 jours : RL vs GINA côte à côte."""
    row = df[df["Diagnosis"]==1][FEATURE_COLS].values[0]
    device = next(agent.q_on.parameters()).device
    baseline = GINABaseline(); baseline.reset()
    agent.q_on.eval()

    def run_episode(use_rl):
        twin = PatientDigitalTwin(row, risk_model, noise=0.03); obs = twin.reset()
        acts_log, act_log, risk_log, fev_log, exacer_log = [], [], [], [], []
        baseline.reset()
        for _ in range(CFG["twin_horizon"]):
            if use_rl:
                with torch.no_grad():
                    a = agent.q_on(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
            else:
                apt = np.mean([obs[i] for i in [20,21,22,23]]) * -25 + 25
                a = baseline.act(obs, apt)
            obs, _, done, info = twin.step(a)
            acts_log.append(a+1); act_log.append(info["act_proxy"])
            risk_log.append(info["risk_score"]); fev_log.append(info["fev1"])
            exacer_log.append(info["exacerbation"])
            if done: break
        return acts_log, act_log, risk_log, fev_log, exacer_log

    rl_data   = run_episode(True)
    gina_data = run_episode(False)
    agent.q_on.train()

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("Digital Twin — Simulation 30 jours (1 patient asthmatique)",
                 fontsize=13, fontweight="bold")
    days = range(1, len(rl_data[0])+1)

    configs = [
        (axes[0,0], rl_data[1], gina_data[1], "Score ACT proxy", "score"),
        (axes[0,1], rl_data[2], gina_data[2], "Score de risque (RF)", "proba"),
        (axes[1,0], rl_data[3], gina_data[3], "FEV1 simulé (L)", "fev"),
        (axes[1,1], rl_data[0], gina_data[0], "Palier GINA recommandé", "step"),
    ]
    for ax, rl_v, gn_v, title, kind in configs:
        ax.plot(days, rl_v, color="#2196F3", lw=2, label="RL Agent", marker="o", ms=3)
        ax.plot(days, gn_v, color="#FF9800", lw=2, label="GINA",     marker="s", ms=3, linestyle="--")
        if kind == "score":
            ax.axhline(20, color="green", linestyle=":", lw=1, alpha=0.7, label="Seuil contrôle (ACT=20)")
        ax.set_title(title); ax.set_xlabel("Jour"); ax.legend(fontsize=8)
        ax.grid(alpha=0.3); ax.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[PLOT] Digital Twin demo → {out}")


# ══════════════════════════════════════════════════════════════════════════════
# 10. PIPELINE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def main(csv_path: str = "asthma_disease_data.csv", out_dir: str = "results"):
    os.makedirs(out_dir, exist_ok=True)

    # ── 1. Chargement & EDA ─────────────────────────────────────────────
    df = load_and_eda(csv_path)
    plot_eda(df, os.path.join(out_dir, "01_eda.png"))

    # 2. Modèle de risque ─────────────────────────────────────────────
    risk_model = AsthmaRiskModel()
    metrics    = risk_model.fit(df)
    plot_risk_model(metrics, os.path.join(out_dir, "02_risk_model.png"))

    # 3. Agent Double DQN ─────────────────────────────────────────────
    state_dim = len(FEATURE_COLS)
    agent = DoubleDQNAgent(state_dim=state_dim, n_actions=CFG["n_actions"])

    # 4. Entraînement ─────────────────────────────────────────────────
    hist = train(agent, df, risk_model, n_episodes=CFG["n_episodes"])
    agent.save(os.path.join(out_dir, "ddqn_digital_twin.pt"))
    plot_training(hist, os.path.join(out_dir, "03_training.png"))

    # 5. Évaluation ───────────────────────────────────────────────────
    results = evaluate(agent, df, risk_model, n_patients=60)
    plot_comparison(results, os.path.join(out_dir, "04_comparison.png"))

    # 6. Simulation Digital Twin ──────────────────────────────────────
    plot_digital_twin_demo(df, agent, risk_model,
                           os.path.join(out_dir, "05_twin_demo.png"))

    # 7. Export JSON des résultats ────────────────────────────────────
    res_path = os.path.join(out_dir, "results.json")
    with open(res_path, "w") as f:
        json.dump({**results, "auc_risk_model": metrics["auc"]}, f, indent=2)

    print(f"\n[DONE] Tous les fichiers dans → {out_dir}/")
    print("  Fichiers générés :")
    for f in ["01_eda.png","02_risk_model.png","03_training.png",
              "04_comparison.png","05_twin_demo.png","ddqn_digital_twin.pt","results.json"]:
        print(f"    {out_dir}/{f}")
    return agent, results, hist


if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--csv",      default="asthma_disease_data.csv")
    p.add_argument("--output",   default="results")
    p.add_argument("--episodes", type=int, default=CFG["n_episodes"])
    args, unknown = p.parse_known_args() # Changed to parse_known_args()
    CFG["n_episodes"] = args.episodes
    main(csv_path=args.csv, out_dir=args.output)


 DATASET : asthma_disease_data.csv
  Lignes : 2,392  |  Colonnes : 29
  Valeurs manquantes : 0
  Asthmatiques (Diagnosis=1) : 124 (5.2%)
  Non-asthmatiques           : 2268 (94.8%)

  Top features corrélées au diagnostic :
    ExerciseInduced                 0.0540
    ChestTightness                  0.0393
    LungFunctionFVC                 0.0296
    Wheezing                        0.0272
    DustExposure                    0.0260
    Coughing                        0.0242
    LungFunctionFEV1                0.0233
    GastroesophagealReflux          0.0228
[PLOT] EDA → results/01_eda.png

[RISK MODEL] RandomForest entraîné
  AUC ROC   : 0.9958
  Accuracy  : 0.9736
  Top 5 features : ['ChestTightness', 'ExerciseInduced', 'NighttimeSymptoms', 'Coughing', 'Wheezing']
[PLOT] Risk model → results/02_risk_model.png

[AGENT] Double DQN | state_dim=26 | device=cuda | params=49,285

[TRAIN] 600 épisodes | 124 patients asthmatiques
  Ép  100/600 | R_moy=+36.45 | ACT=22.6 | Exacer=0.165 | ε=0